## Register the Tableflow REST Catalog as the default Spark catalog

When you run this cell, you'll be prompted for the REST Catalog endpoint, API key, and API secret you copied in Step 1 of the Part 2 README. The secret is read with `getpass`, so it is never written into the notebook source.

In [ ]:
import getpass
import pyspark
from pyspark.sql import SparkSession

catalog_uri = input('Tableflow REST Catalog endpoint: ')
api_key = input('Tableflow API key: ')
api_secret = getpass.getpass('Tableflow API secret: ')

conf = (
    pyspark.SparkConf()
        .setAppName('Jupyter')
        .set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .set("spark.sql.catalog.tableflowdemo", "org.apache.iceberg.spark.SparkCatalog")
        .set("spark.sql.catalog.tableflowdemo.type", "rest")
        .set("spark.sql.catalog.tableflowdemo.uri", catalog_uri)
        .set("spark.sql.catalog.tableflowdemo.credential", f"{api_key}:{api_secret}")
        .set("spark.sql.catalog.tableflowdemo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .set("spark.sql.catalog.tableflowdemo.rest-metrics-reporting-enabled", "false")
        .set("spark.sql.catalog.tableflowdemo.s3.remote-signing-enabled", "true")
        .set("spark.sql.defaultCatalog", "tableflowdemo")
)
spark = SparkSession.builder.config(conf=conf).getOrCreate()

## List the tables Tableflow has published for this cluster

Replace `<your-kafka-cluster-id>` with your cluster ID (the `lkc-...` value, not the cluster name).

In [ ]:
%%sql
SHOW TABLES in `<your-kafka-cluster-id>`

## Query the stock-trades Iceberg table

This is the same data you produced with the Datagen connector in Part 1, except now it's a queryable Iceberg table, kept current by Tableflow, with no pipeline you had to build.

In [ ]:
%%sql
SELECT *
FROM `<your-kafka-cluster-id>`.`stock-trades`
LIMIT 20;

## Re-run the cell above after a minute

The Datagen connector from Part 1 is still producing trades. Re-running the query above shows new rows without you having done anything. Tableflow keeps the table current as the topic grows.